In [1]:
import pandas as pd

In [4]:
df= pd.read_csv("course_lead_scoring.csv")

## **Data Preparation**

- Check if the missing values are presented in the features.
- If there are missing values:
  - For categorical features, replace them with 'NA'
  - For numerical features, replace with with 0.0

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1462 entries, 0 to 1461
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               1334 non-null   object 
 1   industry                  1328 non-null   object 
 2   number_of_courses_viewed  1462 non-null   int64  
 3   annual_income             1281 non-null   float64
 4   employment_status         1362 non-null   object 
 5   location                  1399 non-null   object 
 6   interaction_count         1462 non-null   int64  
 7   lead_score                1462 non-null   float64
 8   converted                 1462 non-null   int64  
dtypes: float64(2), int64(3), object(4)
memory usage: 102.9+ KB


In [6]:
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object', 'category']).columns

In [7]:
df[num_cols] = df[num_cols].fillna(0.0)
df[cat_cols] = df[cat_cols].fillna('NA')

In [8]:
print(df.isnull().sum())

lead_source                 0
industry                    0
number_of_courses_viewed    0
annual_income               0
employment_status           0
location                    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64


## **Q1**
What is the most frequent observation (mode) for the column industry?

- NA
- technology
- healthcare
- retail

In [9]:
mode_industry = df['industry'].mode()[0]
print("Most frequent industry:", mode_industry)

Most frequent industry: retail


## **Q2**

- Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- interaction_count and lead_score
- number_of_courses_viewed and lead_score
- number_of_courses_viewed and interaction_count
- annual_income and interaction_count

Only consider the pairs above when answering this question.

In [36]:
num_df = df.select_dtypes(include=['int64', 'float64'])
corr_matrix = num_df.corr()
print(corr_matrix)

                          number_of_courses_viewed  annual_income  \
number_of_courses_viewed                  1.000000       0.009770   
annual_income                             0.009770       1.000000   
interaction_count                        -0.023565       0.027036   
lead_score                               -0.004879       0.015610   
converted                                 0.435914       0.053131   

                          interaction_count  lead_score  converted  
number_of_courses_viewed          -0.023565   -0.004879   0.435914  
annual_income                      0.027036    0.015610   0.053131  
interaction_count                  1.000000    0.009888   0.374573  
lead_score                         0.009888    1.000000   0.193673  
converted                          0.374573    0.193673   1.000000  


In [37]:
pairs = [
    ('interaction_count', 'lead_score'),
    ('number_of_courses_viewed', 'lead_score'),
    ('number_of_courses_viewed', 'interaction_count'),
    ('annual_income', 'interaction_count')
]
for a, b in pairs:
    print(f"Correlation between {a} and {b}: {corr_matrix.loc[a, b]:.3f}")

Correlation between interaction_count and lead_score: 0.010
Correlation between number_of_courses_viewed and lead_score: -0.005
Correlation between number_of_courses_viewed and interaction_count: -0.024
Correlation between annual_income and interaction_count: 0.027


## **Split the data**
- Split your data in train/val/test sets with 60%/20%/20% distribution.
- Use Scikit-Learn for that (the train_test_split function) and set the seed to 42.
- Make sure that the target value converted is not in your dataframe.

In [38]:
from sklearn.model_selection import train_test_split

In [39]:
X = df.drop(columns=['converted'])
y = df['converted']

In [40]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

In [41]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

## **Q3**
- Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
- Round the scores to 2 decimals using round(score, 2).

Which of these variables has the biggest mutual information score?
- industry
- location
- lead_source
- employment_status

In [42]:
from sklearn.feature_selection import mutual_info_classif

In [43]:
cat_cols = X_train.select_dtypes(include=['object', 'category']).columns

X_train_encoded = X_train[cat_cols].apply(lambda x: x.astype('category').cat.codes)

mi_scores = mutual_info_classif(X_train_encoded, y_train, random_state=42)

mi_df = pd.DataFrame({
    'feature': cat_cols,
    'mutual_info': [round(score, 2) for score in mi_scores]
}).sort_values(by='mutual_info', ascending=False)

print(mi_df)


             feature  mutual_info
0        lead_source         0.03
1           industry         0.00
2  employment_status         0.00
3           location         0.00


## **Q4**
- Now let's train a logistic regression.
- Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
- Fit the model on the training dataset.
  - To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
  - model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
- Calculate the accuracy on the validation dataset and round it to 2 decimal digits.

What accuracy did you get?

- 0.64
- 0.74
- 0.84
- 0.94

In [19]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [20]:
X_train_enc = pd.get_dummies(X_train, drop_first=True)
X_val_enc = pd.get_dummies(X_val, drop_first=True)

X_val_enc = X_val_enc.reindex(columns=X_train_enc.columns, fill_value=0)

In [21]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_enc, y_train)

LogisticRegression(max_iter=1000, random_state=42, solver='liblinear')

In [22]:
y_pred = model.predict(X_val_enc)
acc = accuracy_score(y_val, y_pred)
print("Validation accuracy:", round(acc, 2))

Validation accuracy: 0.68


## **Q5**
- Let's find the least useful feature using the feature elimination technique.
- Train a model using the same features and parameters as in Q4 (without rounding).
- Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
- For each feature, calculate the difference between the original accuracy and the accuracy without the feature.

Which of following feature has the smallest difference?
- 'industry'
- 'employment_status'
- 'lead_score'

Note: The difference doesn't have to be positive.



In [23]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
model.fit(X_train_enc, y_train)
y_pred = model.predict(X_val_enc)
base_acc = accuracy_score(y_val, y_pred)
print("Baseline accuracy:", base_acc)

Baseline accuracy: 0.684931506849315


In [24]:
features_to_check = ['industry', 'employment_status', 'lead_score']
diffs = {}

for feat in features_to_check:

    cols_to_drop = [c for c in X_train_enc.columns if feat in c]
    
    X_train_drop = X_train_enc.drop(columns=cols_to_drop)
    X_val_drop = X_val_enc.drop(columns=cols_to_drop)
    
    model.fit(X_train_drop, y_train)
    y_pred = model.predict(X_val_drop)
    acc = accuracy_score(y_val, y_pred)
    
    diffs[feat] = base_acc - acc

for feat, diff in diffs.items():
    print(f"{feat}: {diff:.4f}")

industry: 0.0000
employment_status: 0.0034
lead_score: 0.0068


## **Q6**
- Now let's train a regularized logistic regression.
- Let's try the following values of the parameter C: [0.01, 0.1, 1, 10, 100].
- Train models using all the features as in Q4.
- Calculate the accuracy on the validation dataset and round it to 3 decimal digits.

Which of these C leads to the best accuracy on the validation set?

Note: If there are multiple options, select the smallest C.

In [26]:
X_train_enc = pd.get_dummies(X_train, drop_first=True)
X_val_enc = pd.get_dummies(X_val, drop_first=True)
X_val_enc = X_val_enc.reindex(columns=X_train_enc.columns, fill_value=0)

In [27]:
C_values = [0.01, 0.1, 1, 10, 100]
accuracies = {}

for c in C_values:
    model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
    model.fit(X_train_enc, y_train)
    y_pred = model.predict(X_val_enc)
    acc = accuracy_score(y_val, y_pred)
    accuracies[c] = round(acc, 3)

# Print hasil
for c, acc in accuracies.items():
    print(f"C={c}: validation accuracy = {acc}")

C=0.01: validation accuracy = 0.688
C=0.1: validation accuracy = 0.682
C=1: validation accuracy = 0.685
C=10: validation accuracy = 0.685
C=100: validation accuracy = 0.685
